# 🚀 Baidu Unlimited-OCR FastAPI Microservice on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arslaan7861/unlimited-ocr/blob/master/colab_unlimited_ocr.ipynb)

This notebook runs the **Baidu Unlimited-OCR** FastAPI microservice directly on Google Colab GPU.
- **Model Source**: Directly downloaded from Hugging Face (`baidu/Unlimited-OCR`).
- **Public Access**: Exposed via **Cloudflare Tunnel (`cloudflared`)** (no accounts, tokens, or bandwidth limits).

### Instructions:
1. **Set Runtime to GPU**: Click `Runtime` -> `Change runtime type` -> Select `T4 GPU` (or `A100` / `L4`) -> Click `Save`.
2. **Run All**: Click `Runtime` -> `Run all` (or press `Ctrl+F9`).

### Step 1: Verify NVIDIA GPU Acceleration

In [ ]:
!nvidia-smi

### Step 2: Clone Repository

In [ ]:
import os

if not os.path.exists("app"):
    !git clone https://github.com/arslaan7861/unlimited-ocr.git
    %cd unlimited-ocr
else:
    print("Already inside repository root.")

### Step 3: Install Required Dependencies & Cloudflared

In [ ]:
# Install web framework, PyMuPDF, transformers, and model helper libraries
!pip install -q fastapi "uvicorn[standard]" pydantic-settings python-multipart pymupdf transformers accelerate matplotlib addict einops easydict

# Download official Cloudflare Tunnel binary
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
print("✅ Dependencies and cloudflared installed successfully.")

### Step 4: Configure Production Environment (.env)

In [ ]:
env_content = """# Google Colab GPU Configuration
HOST=0.0.0.0
PORT=8000
ENVIRONMENT=production
LOG_LEVEL=INFO
MODEL_PATH=baidu/Unlimited-OCR
DEVICE=cuda
TORCH_DTYPE=bfloat16
MAX_UPLOAD_SIZE_MB=25
MAX_PDF_PAGES=50
MAX_CONCURRENT_INFERENCES=4
PDF_DPI=300
"""

with open(".env", "w", encoding="utf-8") as f:
    f.write(env_content)

print("✅ .env configured to use baidu/Unlimited-OCR from Hugging Face with CUDA.")

### Step 5: Download & Cache Model Weights from Hugging Face

Downloading the ~5 GB weights here shows the progress bar directly in Colab so you know when the model is ready before starting the server.

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer

print("⏳ Downloading Baidu Unlimited-OCR weights from Hugging Face (approx. 1-2 mins on Colab)...")
AutoTokenizer.from_pretrained("baidu/Unlimited-OCR", trust_remote_code=True)
AutoModel.from_pretrained(
    "baidu/Unlimited-OCR",
    trust_remote_code=True,
    use_safetensors=True,
    torch_dtype=torch.bfloat16,
)
print("✅ Baidu Unlimited-OCR model cached successfully in Hugging Face cache!")

### Step 6: Start FastAPI Server & Launch Cloudflare Tunnel

Now that the weights are pre-cached, the FastAPI server will boot in under 5 seconds and will be ready for requests immediately.

In [ ]:
import re
import subprocess
import time

# 1. Launch Cloudflare Tunnel connected to IPv4 127.0.0.1:8000
tunnel_proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

# 2. Extract the public HTTPS trycloudflare.com URL
public_url = None
for _ in range(30):
    line = tunnel_proc.stderr.readline().decode()
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break
    time.sleep(0.5)

print("=" * 65)
if public_url:
    print(f"🚀 PUBLIC API URL:     {public_url}")
    print(f"📖 INTERACTIVE DOCS:   {public_url}/docs")
    print(f"🩺 HEALTH PROBE:       {public_url}/health")
else:
    print("Cloudflare tunnel started. If URL not detected above, check stderr.")
print("=" * 65)

# 3. Run FastAPI microservice
!uvicorn app.main:app --host 0.0.0.0 --port 8000